# FairPlay AI: detección de clones de videojuegos con DBSCAN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lina-cc/sic-ia-clone-radar/blob/main/FairPlay_AI_DBSCAN.ipynb)

> **Nota (actualización):** este fue el primer experimento. Las imágenes eran *simuladas* y los clones compartían un centro visual por construcción, así que el resultado muestra que combinar señales ayuda *si la imagen es informativa*, pero no mide el rendimiento con juegos reales. El trabajo con datos reales de Steam está en `FairPlay_AI_Clasificador_Pares.ipynb`.

Integrantes del equipo: Ignacio Hidalgo, Sergio Rebolledo, Catalina Caris, Avelyn García.

-----

Ejercicio con **datos sintéticos** (descripciones + embeddings de imagen simulados).

- **Línea base:** TF-IDF de descripciones + DBSCAN
- **Modelo base:** distancia combinada (texto + imagen) + DBSCAN

> Un cluster indica similitud técnica, no plagio. La decisión final es de un especialista humano.

-----

## Frase de Valor

> Ayudamos a desarrolladores independientes a identificar y documentar patrones de réplica de sus videojuegos en plataformas digitales mediante el análisis comparativo de metadatos, elementos visuales y descripciones con inteligencia artificial a diferencia de la revisión manual y lentos procesos de reclamo tradicionales, nuestro sistema entrega reportes técnicos objetivos y reproducibles que facilitan la evaluación humana y el respaldo antes plataformas.

## Test Card

**Creemos que:** un desarrollador indie puede encontrar juegos que copien al suyo comparando sus descripciones y sus captura de imágenes.

**Para verificarlo, vamos a:** construir un modelo que combina la distancia de texto y la imagen, lo agrupa con DBSCAN y lo copara con la línea base que solo usa texto.

**Mediremos:** cuántos de los 15 juegos de familias clon agrupa con su familia (recall) y cuántos de los 13 juegos independientes marca por error como clon (falsas alarmas), además de la precisión por pares.

**Tenemos razón si:** el modelo agrupa al menos 12 de los 15 juegos de la familia, marca a lo más 1 de los independientes, y tiene menos falsas alarmas que el base.



In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

import zlib
import gradio as gr


## 1. Dataset sintético
3 grupos de clones (*Escape the Walls*, *Farm Tycoon*, *Racing Legends*) y 13 juegos independientes, incluyendo señuelos del mismo género.

In [2]:
rng = np.random.default_rng(42)
D_IMG = 32  # dimensión de los "embeddings CNN" simulados

grupos = {
    0: ("Escape the Walls",
        ["escapa de los muros que se cierran", "esquiva obstaculos",
         "sobrevive hasta llegar a la salida", "corre por pasillos que cambian"],
        ["Escape Walls Obby", "Walls Escape 3D", "Escape the Walls Clone", "Run From Walls",
         "Wall Runner Escape", "Escape the Wall Maze", "Walls Obby Escape"]),
    1: ("Farm Tycoon",
        ["construye tu granja", "cultiva y vende productos",
         "mejora tus herramientas", "contrata ayudantes"],
        ["Farm Tycoon Pro", "My Farm Empire", "Farm Tycoon Remake"]),
    2: ("Racing Legends",
        ["carreras de autos", "mejoras para tu vehiculo",
         "circuitos urbanos", "compite con otros jugadores"],
        ["Racing Legends 2", "Street Racing Legends"]),
}

# Juegos independientes (grupo -1). Incluye señuelos del mismo género que el grupo 0.
independientes = {
    "Obby Escape Tower": "sube una torre con saltos dificiles, evita lava y llega a la cima",
    "Escape the Lab": "resuelve acertijos en un laboratorio y encuentra la llave para salir",
    "Wall Climb Obby": "escapa de la lava que sube, esquiva obstaculos y sobrevive hasta la cima",
    "Escape the Prison": "escapa de una carcel, esquiva guardias y sobrevive hasta llegar a la salida",
    "Maze Runner Obby": "corre por pasillos de un laberinto, evita trampas y llega a la salida",
    "Pet Simulator Mini": "adopta mascotas, evoluciónalas y completa misiones diarias",
    "Zombie Siege": "defiende tu base de oleadas de zombies con armas mejoradas",
    "Cooking Chaos": "prepara pedidos rapidos en una cocina con tiempo limite",
    "Dino Park Builder": "construye un parque de dinosaurios y cuida a los visitantes",
    "Sky Parkour": "salta entre islas flotantes y desbloquea trucos de parkour",
    "Space Miner": "mina asteroides, vende minerales y mejora tu nave",
    "Hide and Seek Mansion": "escondete en una mansion embrujada mientras un monstruo busca",
    "Pizza Shop Tycoon": "administra tu pizzeria, atiende clientes y expande el local",
}

sinonimos = {"muros": "paredes", "obstaculos": "trampas", "granja": "finca",
             "autos": "coches", "vehiculo": "carro", "escapa": "huye",
             "esquiva": "evita", "sobrevive": "aguanta", "construye": "arma",
             "carreras": "competencias"}

def variar(frases):
    """Simula una descripción reescrita (pocas frases + sinónimos)."""
    f = list(frases)
    rng.shuffle(f)
    f = f[:rng.integers(1, 3)]
    txt = ", ".join(f)
    for a, b in sinonimos.items():
        if rng.random() < 0.6:
            txt = txt.replace(a, b)
    return txt

filas, centros = [], {}
for g, (orig, frases, clones) in grupos.items():
    centros[g] = rng.normal(0, 1, D_IMG)  # "estilo visual" común del grupo
    for nombre in [orig] + clones:
        emb = centros[g] + rng.normal(0, 1.1, D_IMG)  # capturas con ruido
        filas.append((nombre, variar(frases), g, emb))
for nombre, desc in independientes.items():
    filas.append((nombre, desc, -1, rng.normal(0, 1, D_IMG)))

df = pd.DataFrame({"id": range(len(filas)),
                   "titulo": [f[0] for f in filas],
                   "descripcion": [f[1] for f in filas],
                   "grupo_real": [f[2] for f in filas]})
emb = np.array([f[3] for f in filas])
y = df["grupo_real"].values

print("Juegos por grupo real (-1 = sin clones):")
print(df["grupo_real"].value_counts().sort_index(), "\n")

# (Opcional) guardar el dataset como CSV
# pd.concat([df, pd.DataFrame(emb, columns=[f"img_{i}" for i in range(D_IMG)]).round(4)], axis=1)\
#   .to_csv("juegos_sinteticos.csv", index=False)

Juegos por grupo real (-1 = sin clones):
grupo_real
-1    13
 0     8
 1     4
 2     3
Name: count, dtype: int64 



## 2. Matrices de distancia
Coseno sobre TF-IDF (texto), sobre embeddings (imagen) y su promedio.

In [3]:
D_texto = cosine_distances(TfidfVectorizer().fit_transform(df["descripcion"]))  # línea base
D_img = cosine_distances(emb)                                                    # capturas (CNN)
D_comb = 0.5 * D_texto + 0.5 * D_img                                             # modelo base

## 3. Evaluación por pares y barrido de `eps`
Precisión y recall sobre pares de juegos que el modelo agrupa como clones.

In [4]:
def pares_clon(etiquetas):
    return {(i, j) for i, j in combinations(range(len(etiquetas)), 2)
            if etiquetas[i] == etiquetas[j] and etiquetas[i] != -1}

def evaluar(nombre, D, eps, min_samples=2):
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit_predict(D)
    real, pred = pares_clon(y), pares_clon(labels)
    tp = len(real & pred)
    prec = tp / len(pred) if pred else 0
    rec = tp / len(real) if real else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    met = dict(modelo=nombre, eps=round(float(eps), 2), precision=prec, recall=rec,
               F1=f1, ARI=adjusted_rand_score(y, labels), ruido=int((labels == -1).sum()))
    return met, labels

def mejor(nombre, D):
    return max((evaluar(nombre, D, e) for e in np.arange(0.2, 1.0, 0.05)),
               key=lambda t: t[0]["F1"])

(m_base, lab_base) = mejor("Linea base (TF-IDF)", D_texto)
(m_mod, lab_mod) = mejor("Modelo base (texto+imagen)", D_comb)

print("=== MODELO BASE vs LINEA BASE ===")
print(pd.DataFrame([m_base, m_mod]).round(3).to_string(index=False), "\n")

=== MODELO BASE vs LINEA BASE ===
                    modelo  eps  precision  recall    F1   ARI  ruido
       Linea base (TF-IDF) 0.75      0.479   0.946 0.636 0.510     10
Modelo base (texto+imagen) 0.60      1.000   0.838 0.912 0.883     14 



## 4. Tabla de 3 casos probados
Clon evidente, mismo género y sin relación.

In [5]:
idx = lambda n: int(df.index[df.titulo == n][0])

def caso(i, j, nombre):
    return {"caso": nombre, "par": f"{df.titulo[i]} vs {df.titulo[j]}",
            "sim_texto": round(1 - D_texto[i, j], 2),
            "sim_imagen": round(1 - D_img[i, j], 2),
            "sim_combinada": round(1 - D_comb[i, j], 2),
            "mismo_cluster_base": bool(lab_base[i] == lab_base[j] and lab_base[i] != -1),
            "mismo_cluster_modelo": bool(lab_mod[i] == lab_mod[j] and lab_mod[i] != -1)}

tabla = pd.DataFrame([
    caso(idx("Escape the Walls"), idx("Escape Walls Obby"), "1 Clon evidente"),
    caso(idx("Escape the Walls"), idx("Escape the Lab"), "2 Mismo genero"),
    caso(idx("Escape the Walls"), idx("Farm Tycoon"), "3 Sin relacion"),
])
print("=== 3 CASOS PROBADOS ===")
print(tabla.to_string(index=False))
print("\nNota: un cluster indica similitud tecnica, no plagio. La decision final es de un especialista humano.")

=== 3 CASOS PROBADOS ===
           caso                                   par  sim_texto  sim_imagen  sim_combinada  mismo_cluster_base  mismo_cluster_modelo
1 Clon evidente Escape the Walls vs Escape Walls Obby        0.0        0.52           0.26                True                  True
 2 Mismo genero    Escape the Walls vs Escape the Lab        0.0       -0.04          -0.02               False                 False
 3 Sin relacion       Escape the Walls vs Farm Tycoon        0.0        0.33           0.16               False                 False

Nota: un cluster indica similitud tecnica, no plagio. La decision final es de un especialista humano.


## Comprobación del umbral

In [6]:
# ── Test Card: ¿cumplimos el umbral? ─────────────────────────────────────────
MIN_AGRUPADOS = 12   # de los juegos que pertenecen a una familia clon (15)
MAX_FALSAS    = 1    # de los juegos independientes (13), a lo más 1 marcado como clon

def evaluar_juegos(D, eps, min_samples=2):
    """Cuenta juegos (no pares): aciertos de familia y falsas alarmas."""
    lab = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit_predict(D)
    familia = np.where(y != -1)[0]   # juegos que pertenecen a una familia clon
    indep   = np.where(y == -1)[0]   # juegos independientes (incluye señuelos)
    # acierto: el juego quedó en un cluster junto a otro juego de SU MISMA familia
    agrupados = [i for i in familia if lab[i] != -1
                 and any(lab[j] == lab[i] and y[j] == y[i] for j in familia if j != i)]
    # falsa alarma: un independiente quedó dentro de algún cluster
    falsas = [i for i in indep if lab[i] != -1]
    return len(agrupados), len(familia), len(falsas), len(indep), [df.titulo[i] for i in falsas]

filas = []
for nombre, D, met in [("Línea base (solo texto)", D_texto, m_base),
                       ("Modelo (texto + imagen)", D_comb, m_mod)]:
    ok, n_fam, fa, n_ind, quienes = evaluar_juegos(D, met["eps"])
    filas.append({
        "sistema": nombre,
        "eps": met["eps"],
        "familia_agrupada": f"{ok} de {n_fam}",
        "falsas_alarmas": f"{fa} de {n_ind}",
        "cumple_recall": ok >= MIN_AGRUPADOS,
        "cumple_falsas": fa <= MAX_FALSAS,
        "independientes_marcados": quienes,
    })
res = pd.DataFrame(filas)
print(res.to_string(index=False))

modelo, base = filas[1], filas[0]
fa_modelo = int(modelo["falsas_alarmas"].split()[0]); fa_base = int(base["falsas_alarmas"].split()[0])
mejor_que_base = fa_modelo < fa_base
print("\n=== VEREDICTO DE LA TEST CARD ===")
print(f"Agrupa al menos {MIN_AGRUPADOS} juegos de familia:   {'SÍ' if modelo['cumple_recall'] else 'NO'}  ({modelo['familia_agrupada']})")
print(f"Marca a lo más {MAX_FALSAS} independiente:        {'SÍ' if modelo['cumple_falsas'] else 'NO'}  ({modelo['falsas_alarmas']})")
print(f"Menos falsas alarmas que la línea base: {'SÍ' if mejor_que_base else 'NO'}  ({fa_modelo} vs {fa_base})")
print("-> Tenemos razón:", "SÍ" if modelo["cumple_recall"] and modelo["cumple_falsas"] and mejor_que_base else "NO")

                sistema  eps familia_agrupada falsas_alarmas  cumple_recall  cumple_falsas                                                   independientes_marcados
Línea base (solo texto) 0.75         14 de 15        4 de 13           True          False [Obby Escape Tower, Wall Climb Obby, Escape the Prison, Maze Runner Obby]
Modelo (texto + imagen) 0.60         14 de 15        0 de 13           True           True                                                                        []

=== VEREDICTO DE LA TEST CARD ===
Agrupa al menos 12 juegos de familia:   SÍ  (14 de 15)
Marca a lo más 1 independiente:        SÍ  (0 de 13)
Menos falsas alarmas que la línea base: SÍ  (0 vs 4)
-> Tenemos razón: SÍ


## Robustez del modelo

**Qué problema tiene lo anterior.** Hasta aquí probamos con un solo catálogo simulado
(semilla 42), y el `eps` de cada sistema se eligió mirando ese mismo catálogo. Con solo
28 juegos, un buen resultado podría deberse a que esa tirada salió fácil.

**Qué hacemos.** Generamos 30 catálogos nuevos con la misma receta (mismas familias,
mismos independientes, otras descripciones e imágenes al azar). En todos usamos el
**mismo `eps` de la sección 3**, sin volver a ajustarlo, y medimos lo de la Test Card:
cuántos de los 15 juegos de familia quedan agrupados con su familia, y cuántos de los
13 independientes se marcan por error.

**Cómo leerlo.** Si la Test Card se cumple en casi todos los catálogos, el resultado
no depende de una tirada afortunada. Si se cumple solo en algunos, el `eps` es frágil.

**Qué no demuestra.** Los 30 catálogos salen de la misma receta, así que esto mide
estabilidad frente al azar de la simulación, no funcionamiento con juegos reales.

In [7]:
# ── Robustez: mismo eps, 30 catálogos simulados nuevos ───────────────────────
def generar_catalogo(seed):
    """Misma receta de la sección 1, pero con otra semilla. No toca df, y ni emb."""
    r = np.random.default_rng(seed)
    def variar_s(frases):
        f = list(frases); r.shuffle(f); f = f[:r.integers(1, 3)]
        txt = ", ".join(f)
        for a, b in sinonimos.items():
            if r.random() < 0.6:
                txt = txt.replace(a, b)
        return txt
    filas_s = []
    for g, (orig, frases, clones) in grupos.items():
        centro = r.normal(0, 1, D_IMG)
        for nombre in [orig] + clones:
            emb_i = centro + r.normal(0, 1.1, D_IMG)      # mismo orden de sorteo que la sección 1
            filas_s.append((nombre, variar_s(frases), g, emb_i))
    for nombre, desc in independientes.items():
        filas_s.append((nombre, desc, -1, r.normal(0, 1, D_IMG)))
    desc_s = [f[1] for f in filas_s]
    y_s    = np.array([f[2] for f in filas_s])
    emb_s  = np.array([f[3] for f in filas_s])
    Dt_s = cosine_distances(TfidfVectorizer().fit_transform(desc_s))
    Di_s = cosine_distances(emb_s)
    return y_s, Dt_s, 0.5 * Dt_s + 0.5 * Di_s

def contar_juegos(y_s, D, eps):
    lab = DBSCAN(eps=eps, min_samples=2, metric="precomputed").fit_predict(D)
    fam, ind = np.where(y_s != -1)[0], np.where(y_s == -1)[0]
    ok = sum(1 for i in fam if lab[i] != -1
             and any(lab[j] == lab[i] and y_s[j] == y_s[i] for j in fam if j != i))
    fa = sum(1 for i in ind if lab[i] != -1)
    return ok, fa

N_SEMILLAS = 30
rep = []
for s in range(100, 100 + N_SEMILLAS):          # semillas distintas a la 42
    y_s, Dt_s, Dc_s = generar_catalogo(s)
    ok_b, fa_b = contar_juegos(y_s, Dt_s, m_base["eps"])   # eps FIJO, el de la sección 3
    ok_m, fa_m = contar_juegos(y_s, Dc_s, m_mod["eps"])    # eps FIJO, el de la sección 3
    rep.append({"semilla": s, "base_agrupados": ok_b, "base_falsas": fa_b,
                "modelo_agrupados": ok_m, "modelo_falsas": fa_m})
rep = pd.DataFrame(rep)

cumple = ((rep.modelo_agrupados >= MIN_AGRUPADOS) & (rep.modelo_falsas <= MAX_FALSAS)
          & (rep.modelo_falsas < rep.base_falsas))
print(rep[["base_agrupados", "base_falsas", "modelo_agrupados", "modelo_falsas"]]
      .agg(["mean", "min", "max"]).round(2).to_string())
print(f"\nCatálogos donde el modelo cumple la Test Card completa: {int(cumple.sum())} de {N_SEMILLAS}")

      base_agrupados  base_falsas  modelo_agrupados  modelo_falsas
mean           13.67         4.33             12.93           0.23
min            11.00         4.00             10.00           0.00
max            15.00         5.00             15.00           1.00

Catálogos donde el modelo cumple la Test Card completa: 25 de 30


**Resultado.** El modelo cumple la Test Card en 25 de 30 catálogos. Marca pocas falsas
alarmas (0,2 independientes en promedio, máximo 1), mientras que la línea base marca
4 o 5 en todos. Lo que a veces falla es el recall: en algunos catálogos agrupa menos de
12 de los 15 juegos de familia.

**Conclusión.** La ventaja del modelo sobre la línea base en falsas alarmas es estable.
El recall es más variable y depende de cuánto se parezcan, por azar, los clones de cada
catálogo.

## Prototipo: App

### Variables y funciones para iniciar la app

In [8]:
vec = TfidfVectorizer().fit(df["descripcion"])   # mismo vocabulario que la línea base
X_txt = vec.transform(df["descripcion"])
EPS_BASE, EPS_MOD = m_base["eps"], m_mod["eps"]  # los eps elegidos en la sección 3
SIN_CAPTURAS = "(sin capturas)"
ORIGINAL = "Escape the Walls"
idx_de = lambda t: int(df.index[df.titulo == t][0])


def vecinos(orden, d, eps):
    """Nombres de los juegos a distancia <= eps, del más cercano al más lejano."""
    return [df.titulo.values[i] for i in orden if d[i] <= eps]


def texto_lista(nombres, n=4):
    if not nombres:
        return "ninguno"
    extra = f" (+{len(nombres) - n} más)" if len(nombres) > n else ""
    return ", ".join(nombres[:n]) + extra


def mismo_cluster(d, eps, lab, ref):
    """¿Quedaría el juego en el cluster del juego `ref`? (misma regla que DBSCAN: se une al
    cluster de cualquier juego del catálogo a distancia <= eps; si el juego ya está en el
    catálogo, esto reproduce el cluster que le asignó DBSCAN en la sección 3)."""
    return bool(lab[ref] != -1 and any(d[k] <= eps and lab[k] == lab[ref] for k in range(len(d))))


def analizar(titulo, descripcion, capturas_como, original, exactas):
    descripcion = (descripcion or "").strip()
    if len(descripcion.split()) < 3:
        return "Escribe una descripción de al menos 3 palabras.", pd.DataFrame()
    d_txt = cosine_distances(vec.transform([descripcion]), X_txt).ravel()
    if d_txt.min() > 0.999:
        return ("Ninguna palabra de tu descripción aparece en el dataset. "
                "Usa vocabulario parecido al de los ejemplos (el dataset está en español)."), pd.DataFrame()

    ref = idx_de(original)
    # Si el juego ES uno del catálogo (los ejemplos), no lo contamos como vecino de sí mismo.
    nombre = (titulo or "tu juego").strip() or "tu juego"
    yo = idx_de(nombre) if (nombre in set(df.titulo) and df.descripcion[idx_de(nombre)] == descripcion) else None
    sin_yo = lambda orden: np.array([k for k in orden if k != yo])
    con_img = capturas_como != SIN_CAPTURAS
    if con_img:
        nuevo = emb[idx_de(capturas_como)].copy()   # capturas "como" ese juego
        if not exactas:                              # capturas simuladas = ese juego + ruido propio
            rng_caso = np.random.default_rng(zlib.crc32((descripcion + capturas_como).encode()))
            nuevo = nuevo + rng_caso.normal(0, 1.1, D_IMG)
        d_img = cosine_distances(nuevo.reshape(1, -1), emb).ravel()
        d_comb = 0.5 * d_txt + 0.5 * d_img
        orden = sin_yo(np.argsort(d_comb))
    else:
        d_img = d_comb = None
        orden = sin_yo(np.argsort(d_txt))

    top = orden[:5]
    tabla = pd.DataFrame({
        "Juego": df.titulo.values[top],
        "Txt": (1 - d_txt[top]).round(2),
        "Img": (1 - d_img[top]).round(2) if con_img else ["—"] * 5,
        "Comb": (1 - d_comb[top]).round(2) if con_img else ["—"] * 5,
        "Base": np.where(d_txt[top] <= EPS_BASE, "Sí", "No"),
        "Mod": np.where(d_comb[top] <= EPS_MOD, "Sí", "No") if con_img else ["—"] * 5,
    })

    base_n = vecinos(sin_yo(np.argsort(d_txt)), d_txt, EPS_BASE)
    sb = "Sí" if mismo_cluster(d_txt, EPS_BASE, lab_base, ref) else "No"
    if con_img:
        modelo_n = vecinos(orden, d_comb, EPS_MOD)
        sm = "Sí" if mismo_cluster(d_comb, EPS_MOD, lab_mod, ref) else "No"
        par = (f"sim. texto {1 - d_txt[ref]:.2f} · sim. imagen {1 - d_img[ref]:.2f} · "
               f"sim. combinada {1 - d_comb[ref]:.2f}")
        modelo = texto_lista(modelo_n)
    else:
        sm, modelo = "—", "no se puede evaluar sin capturas (la imagen es parte del modelo)"
        par = f"sim. texto {1 - d_txt[ref]:.2f}"

    # Si el juego es uno del catálogo (ejemplos), mostramos la etiqueta real para poder revisar.
    etiqueta = ""
    if yo is not None:
        g = df.grupo_real[yo]
        etiqueta = ("- **Etiqueta del dataset:** " +
                    (f"es un juego independiente (sin clones)" if g == -1 else
                     f"pertenece a una familia de clones (grupo {g})") + "\n")
    msg = (f"**{nombre}**\n\n{etiqueta}"
           f"- **Línea base (solo texto)** lo agruparía con: {texto_lista(base_n)}\n"
           f"- **Modelo base (texto + imagen)** lo agruparía con: {modelo}\n\n"
           f"**Frente a «{original}»:** {par}\n\n"
           f"- ¿Mismo cluster, línea base? **{sb}**\n"
           f"- ¿Mismo cluster, modelo? **{sm}**\n\n"
           "*Un cluster indica similitud técnica, no plagio. La decisión final es de un especialista humano.*")
    return msg, tabla


def ejemplo(t):
    """Un juego del catálogo, con su descripción y sus capturas tal cual."""
    return [t, df.descripcion[idx_de(t)], t, ORIGINAL, True]

EJEMPLOS = [ejemplo("Escape Walls Obby"),   # 1 clon evidente (misma familia)
            ejemplo("Escape the Lab"),      # 2 mismo género (independiente, señuelo)
            ejemplo("Farm Tycoon")]         # 3 sin relación

### Demo

In [ ]:
try:
    demo.close()          # si re-ejecutas la celda, cierra la app anterior
except NameError:
    pass

demo = gr.Interface(
    fn=analizar,
    inputs=[
        gr.Textbox(label="Título del juego"),
        gr.Textbox(label="Descripción (en español, como en el dataset)", lines=3),
        gr.Dropdown([SIN_CAPTURAS] + list(df.titulo), value=SIN_CAPTURAS,
                    label="Capturas (simuladas): ¿de qué juego se parecen?"),
        gr.Dropdown(list(df.titulo), value=ORIGINAL, label="Juego original con el que comparar"),
        gr.Checkbox(value=False, label="Usar las capturas tal cual, sin ruido (para reproducir casos del dataset)"),
    ],
    outputs=[gr.Markdown(label="Resultado"),
             gr.Dataframe(label="Los 5 juegos más cercanos (Txt, Img, Comb = similitud de texto, imagen y combinada; "
                                "Base y Mod = ¿lo marca la línea base / el modelo?)",
                          wrap=True, column_widths=["36%", "13%", "13%", "14%", "12%", "12%"])],
    title="FairPlay AI: prototipo con datos sintéticos",
    description="Ingresa un juego y compáralo con el catálogo simulado, con la línea base y con el modelo base. "
                "Los 3 ejemplos reproducen la tabla de casos. Las capturas son simuladas. "
                "Un juego nuevo no tiene respuesta conocida: el resultado se revisa a mano.",
    examples=EJEMPLOS,
    cache_examples=False,
)
demo.launch()

## Resumen pruebas

| Caso | Qué ingresamos | Qué predijo | ¿Tiene sentido? ¿Por qué? |
|------|----------------|-------------|---------------------------|
| 1    | Escape the Wals vs Escape the Walls Obby (Clon) | Tanto el cluster de texto como el combinado detectan el clon | Tiene sentido, es un clon creado con el propósito de ser detectado como tal. |
| 2    | Escape the Walls vs Escape the Lab (Mismo género) | Ambos modelos dieron Falso | Tiene sentido, la idea es detectar clones pero siendo sensible a que existen juegos que solo se parecen. |
| 3    | Escape the Walls vs Farm Tycoon (sin relación) | Ambos modelos dieron Falso | Tiene sentido, este es un juego que, si bien tiene clones, no se relación con el juego que estamos comparando. |

## Pitch

1. Gancho: ¿Imaginas pasar meses creando tu videojuego indie y que, días después de publicar tus avances, aparezcan réplicas casi idénticas en Roblox antes de tu estreno oficial? Esto le ocurrió a Pandorio Games con 'Escape the Walls'.

2. Problema: Hoy la IA acelera y abarata la replicación de juegos. Pero no todo parecido es plagio: compartir género o mecánicas es normal. El problema es que no hay una forma simple y transparente de saber cuándo las coincidencias merecen una revisión especializada.

3. Solución: Proponemos FairPlay AI, un sistema experimental asistido por IA que compara las descripciones y las capturas de pantalla de los videojuegos, y señala y explica sus similitudes técnicas. Más adelante podría sumar el código. Lo mostramos en vivo.

4. Evidencia: Simulamos una situación como la de Escape the Walls: 28 juegos, 15 de familias de clones y 13 independientes, varios del mismo género. Un sistema que solo lee descripciones marcó por error a 4 de los 13 independientes. El nuestro, que suma las capturas, no marcó ninguno y reunió a 14 de los 15 clones. Se mantuvo en 25 de 30 catálogos nuevos. Son datos simulados: el siguiente paso es probar con juegos reales.

5. Cierre: No emitimos juicios legales; aportamos evidencia técnica preliminar y transparente para orientar a creadores y expertos. Si la IA facilita copiar, también debe ayudarnos a documentar y proteger la innovación.


## Learning Card
Hecho con la retroalimentación proveida en clases.

**Creíamos que:**
Nuestra presentación y pitch explicaban de manera clara el valor del proyecto mediante el uso de terminología técnica detallada, una estructura narrativa dinámica y el soporte de un dataset simulado como prueba de concepto.

<br>

**Observamos que:**
Lenguaje y Comunicación: El uso excesivo de palabras y conceptos técnicos dificultó que todo el público (especialmente personas no especializadas) comprendiera la propuesta con soltura.
Estructura del Pitch: La narrativa no siguió una línea de tiempo clara; al saltar entre la solución, volver al problema y regresar al inicio, el flujo generó confusión en la audiencia.
Validación de Datos: El público y evaluadores requirieron datos reales de la industria/plataformas para validar el alcance técnico, en lugar de métricas o datasets simulados.

<br>

**De esto aprendimos que:**
Es fundamental simplificar el lenguaje y usar analogías accesibles para transmitir el impacto del proyecto sin perder rigor técnico.
Un pitch efectivo debe mantener una narrativa lineal y fluida (Problema => Solución => Validación => Cierre) para no perder la atención ni la claridad.
Para dar solidez a la propuesta ante evaluadores, la validación debe sustentarse en datos reales extraídos directamente de plataformas o casos reales estudiados (como los de la tienda de Steam o Kaggle).

<br>

**Por lo tanto, vamos a [ X ] Pivotar        [   ] Perseverar        [   ] Archivar**

Porque: Debemos ajustar la estrategia de comunicación y la fase de validación de datos. Mantendremos el núcleo técnico del proyecto (IA para detección de similitudes entre videojuegos), pero reformularemos el guión del pitch hacia una estructura lineal y empática, al mismo tiempo que reemplazaremos los datos simulados por un pipeline de prueba alimentado con datos y métricas reales.

## Prompts

|Herramienta    |Prompt|¿Para qué sirvió?|¿Lo modificamos?|
|---------------|------|-----------------|----------------|
|Antigravity IDE|Analiza el informe_contexto.docx y el prototipo del proyecto<br> para redactar la hoja de trabajo Capstone y pitch ajustado a <br>los tiempos de la Clase 9 de SIC 2026. |Articular la coherencia entre problemática de la industria, <br>solución técnica y guion de presentación. |Sí, se adaptaron las métricas del Test Card y <br>la duración en segundos de cada momento del pitch.|
|Claude         |Genera un Pitch de 5 minutos. | Para recapitular las ideas para el pitch.   | Si, adaptamos algunas palabras para ajustarnos al tiempo.
|Claude         | Crea un md del pitch. | Para tener un documento que compartir en discord | No, el texto se uso como tal para compartir.